# Curated mixed-granularity attention
 
 This screen compares a receptor mean ESM vector (1280-d) against variable-length per-site GIN molecule embeddings (300-d per site). The attention model uses the protein token as a query over molecule sites. Results are cached by regime, model and seed under `results/attention/curated/site_global/`.
 
 Regimes: `transductive` uses independent stratified pair splits; `inductive_molecule` holds out complete molecules. The launcher runs both regimes on seeds 42, 43, 44, 45 and 46. Controls: mean-pooled GIN + ESM XGBoost, and a masked mean/max pooling MLP. The shell queue supports `MAX_PARALLEL`, live status files, logs and a dashboard, while keeping per-seed outputs isolated and merging them for this notebook.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
OUT = ROOT / 'results/attention/curated/site_global'
METRICS = OUT / 'metrics.csv'
print('results:', OUT)


In [ ]:
if not METRICS.exists():
    raise FileNotFoundError('Run legacy/scripts/queues/run_curated_site_global_attention.sh first')
results = pd.read_csv(METRICS)
results

In [ ]:
# Mean and 95% bootstrap-free seed intervals (normal approximation over independent seeds).
summary = (results.groupby(['regime','model'])
           .agg(n=('seed','count'), AUROC=('AUROC','mean'), AUROC_sd=('AUROC','std'),
                AUPRC=('AUPRC','mean'), AUPRC_sd=('AUPRC','std'), MCC=('MCC','mean'),
                MCC_sd=('MCC','std'), F1=('F1','mean'), F1_sd=('F1','std'))
           .reset_index())
for metric in ['AUROC','AUPRC','MCC','F1']:
    summary[metric + '_lo'] = summary[metric] - 1.96*summary[metric+'_sd'].fillna(0)/np.sqrt(summary.n)
    summary[metric + '_hi'] = summary[metric] + 1.96*summary[metric+'_sd'].fillna(0)/np.sqrt(summary.n)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, metric in zip(axes, ['AUROC','AUPRC']):
    labels, vals, lo, hi = [], [], [], []
    for _, row in summary.iterrows():
        labels.append(f"{row.regime}\n{row.model}")
        vals.append(row[metric]); lo.append(row[metric]-row[metric+'_lo']); hi.append(row[metric+'_hi']-row[metric])
    ax.bar(np.arange(len(vals)), vals, yerr=[lo,hi], capsize=4)
    ax.set_xticks(np.arange(len(vals)), labels, rotation=35, ha='right')
    ax.set_ylabel(metric); ax.set_ylim(0, 1); ax.grid(axis='y', alpha=.25)
    ax.set_title(f'{metric}: mean В± 95% CI')
plt.show()

In [ ]:
# Per-seed ROC and PR curves, one separate figure per (regime, model) pair --
# both regimes included, each plotted on its own so individual curves stay readable
# instead of being overlaid together in a cramped shared subplot.
for regime in ['transductive', 'inductive_molecule']:
    for model in sorted(results.loc[results.regime == regime, 'model'].unique()):
        group = results[(results.regime == regime) & (results.model == model)]
        fig, (axr, axp) = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
        for _, row in group.iterrows():
            path = OUT / f"pred_{regime}_{model}_seed{int(row.seed)}.npz"
            if not path.exists(): continue
            z = np.load(path)
            fpr, tpr, _ = roc_curve(z['y'], z['p'])
            prec, rec, _ = precision_recall_curve(z['y'], z['p'])
            axr.plot(fpr, tpr, alpha=.6, label=f"seed {int(row.seed)}")
            axp.plot(rec, prec, alpha=.6, label=f"seed {int(row.seed)}")
        axr.plot([0, 1], [0, 1], ls='--', lw=1, color='grey', alpha=.6)
        axr.set_title('ROC'); axr.set_xlabel('FPR'); axr.set_ylabel('TPR')
        axp.set_title('PR'); axp.set_xlabel('Recall'); axp.set_ylabel('Precision')
        for ax in (axr, axp):
            ax.grid(alpha=.25); ax.legend(fontsize=7)
        fig.suptitle(f'{regime} / {model}')
        plt.show()

## Attention excess max-score constraint
 
 This add-on treats the attention distribution itself as the predictor, but subtracts the uniform baseline first. The saved score is `p = (max(attention_weights) - 1/N) / (1 - 1/N)`, so 0 means uniform attention and 1 means all mass on one molecule site. Negative pairs are penalized when this excess score exceeds `neg_threshold`; positive pairs are penalized when it stays below `pos_threshold`. Run it with `legacy/scripts/queues/run_curated_site_attention_max.sh`.

In [ ]:
# Optional constrained attention-as-score experiment.
SITE_MAX = ROOT / 'results/attention/curated/site_max'
SITE_MAX_METRICS = SITE_MAX / 'metrics.csv'

if SITE_MAX_METRICS.exists():
    max_results = pd.read_csv(SITE_MAX_METRICS)
    display(max_results)
else:
    max_results = pd.DataFrame()
    print('site-max attention results not found yet; run legacy/scripts/queues/run_curated_site_attention_max.sh')


In [ ]:
if not max_results.empty:
    plot_metrics = ['AUROC', 'AUPRC', 'MCC', 'F1']
    summary_max = (max_results
        .groupby(['regime', 'model'])[plot_metrics]
        .agg(['mean', 'std', 'count']))
    display(summary_max.round(3))

    fig, axes = plt.subplots(1, len(plot_metrics), figsize=(14, 4), constrained_layout=True)
    for ax, metric in zip(axes, plot_metrics):
        rows = []
        for (regime, model), g in max_results.groupby(['regime', 'model']):
            vals = g[metric].dropna().astype(float)
            mean = vals.mean()
            sem95 = 1.96 * vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0
            rows.append((f'{regime}\n{model}', mean, sem95))
        labels, means, errs = zip(*rows)
        ax.bar(range(len(means)), means, yerr=errs, capsize=4)
        ax.set_title(metric)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=0, fontsize=8)
        ax.set_ylim(0, 1)
    fig.suptitle('Constrained attention max-score: mean ± 95% CI over seeds')
    plt.show()


### Research note: ensemble candidate
 
 The excess max-attention run is unexpectedly useful despite being deliberately constrained and somewhat hacky. On the current 5-seed screen it reached approximately AUROC/AUPRC = 0.840/0.398 in transductive mode and 0.654/0.216 in inductive-molecule mode.
 
 Interpretation caveat: the older `site_cross` setup is ProSmith-like in spirit because it uses site/token-level cross-attention between molecular sites and protein information, but it should not be described as literally ProSmith unless we match their exact inputs, architecture and evaluation protocol.
 
 Working hypothesis for the next ensemble stage: use the constrained attention score, especially `excess_max_attention`, as an additional feature for the boosting ensemble. It may capture a sparse site-level interaction signal that is complementary to the standard mean-embedding boost and could help close or exceed the ProSmith comparison when evaluated under the same protocol.

In [ ]:
# Distribution of the actual score p=excess max attention over uniform, useful for checking
# whether thresholds are doing what we wanted.
if not max_results.empty:
    pred_files = sorted(SITE_MAX.glob('pred_*.npz'))
    rows = []
    for path in pred_files:
        regime = 'inductive_molecule' if path.stem.startswith('pred_inductive_molecule') else 'transductive'
        arr = np.load(path)
        raw = arr['raw_max_attention'] if 'raw_max_attention' in arr.files else np.full_like(arr['p'], np.nan)
        for label, score, raw_score in zip(arr['y'], arr['p'], raw):
            rows.append({
                'regime': regime,
                'label': int(label),
                'excess_max_attention': float(score),
                'raw_max_attention': float(raw_score),
            })
    score_df = pd.DataFrame(rows)
    if not score_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True, sharey=True)
        for ax, (regime, g) in zip(axes, score_df.groupby('regime')):
            ax.hist(g.loc[g.label == 0, 'excess_max_attention'], bins=30, alpha=0.65, density=True, label='negative')
            ax.hist(g.loc[g.label == 1, 'excess_max_attention'], bins=30, alpha=0.65, density=True, label='positive')
            neg_t = float(max_results['neg_threshold'].iloc[0])
            pos_t = float(max_results['pos_threshold'].iloc[0])
            ax.axvline(neg_t, ls='--', lw=1, color='tab:blue', label=f'neg_t={neg_t:g}')
            ax.axvline(pos_t, ls='--', lw=1, color='tab:orange', label=f'pos_t={pos_t:g}')
            ax.set_title(regime)
            ax.set_xlabel('excess max attention over uniform')
            ax.legend(fontsize=8)
        axes[0].set_ylabel('density')
        fig.suptitle('Score distribution: p = excess max attention over uniform')
        plt.show()
